# Final Demo: Dataset Streaming Simulation

This notebook is the final presentation layer for the dataset-only demo. It uses downloaded ARCO-ERA5 weather and BTS flight outcomes, then runs a historical test-set replay where Gold feature rows from the downloaded lakehouse are published to Kafka for streaming evidence and separately scored by the deployed prediction API.

All prediction evidence comes from the downloaded project dataset and generated lakehouse outputs.

Rerun safety: this notebook does not rewrite raw or lakehouse tables. It overwrites only the local JSONL evidence file for this notebook run, publishes replay messages to Kafka, calls the prediction API, and increases Prometheus/Grafana request metrics. That is expected for a live final demo and does not damage the infrastructure.


In [1]:
from pathlib import Path
import json
import os
import subprocess
import textwrap

PROJECT_ROOT = Path('/workspace') if Path('/workspace').exists() else Path.cwd().resolve()
STREAM_DIR = PROJECT_ROOT / 'data/local_cache/streaming_predictions'
STREAM_DIR.mkdir(parents=True, exist_ok=True)

print('Project root:', PROJECT_ROOT)
print('Streaming evidence dir:', STREAM_DIR)

Project root: /workspace
Streaming evidence dir: /workspace/data/local_cache/streaming_predictions


## 1. Helper Functions

These helpers keep command output visible in the notebook while preserving commands exactly as they can be rerun from the VM or Jupyter container.


In [2]:
def run(command: str, timeout: int = 120) -> str:
    print('$', command)
    completed = subprocess.run(
        command,
        shell=True,
        cwd=PROJECT_ROOT,
        text=True,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        timeout=timeout,
    )
    print(completed.stdout)
    if completed.returncode != 0:
        raise RuntimeError(f'Command failed with exit code {completed.returncode}')
    return completed.stdout


def tail_jsonl(path: Path, limit: int = 5):
    if not path.exists():
        print('Missing JSONL:', path)
        return []
    rows = [json.loads(line) for line in path.read_text().splitlines() if line.strip()]
    return rows[-limit:]

## 2. Service Health

The final demo needs MinIO, Kafka, Spark, MLflow, BentoML API, Prometheus, Grafana, and optionally the LLM chat UI. These checks use service APIs/Python clients so the notebook runs directly from the Jupyter container.


In [3]:

import requests

service_checks = {
    'aviation-api metrics': 'http://aviation-api:3000/metrics',
    'prometheus readiness': 'http://prometheus:9090/-/ready',
    'grafana health': 'http://grafana:3000/api/health',
    'kafka-ui': 'http://kafka-ui:8080',
    'mlflow': 'http://mlflow:5000',
    'spark-master json': 'http://spark-master:8080/json/',
    'llm-chat health': 'http://llm-chat:7860/api/health',
}

for name, url in service_checks.items():
    try:
        response = requests.get(url, timeout=10)
        print(f'{name}: HTTP {response.status_code}')
        if name == 'spark-master json' and response.ok:
            spark_status = response.json()
            print({
                'spark_status': spark_status.get('status'),
                'alive_workers': spark_status.get('aliveworkers'),
                'cores': spark_status.get('cores'),
                'active_apps': len(spark_status.get('activeapps', [])),
            })
    except Exception as error:
        print(f'{name}: {error}')


aviation-api metrics: HTTP 200
prometheus readiness: HTTP 200
grafana health: HTTP 200
kafka-ui: HTTP 200
mlflow: HTTP 200
spark-master json: HTTP 200
{'spark_status': 'ALIVE', 'alive_workers': 1, 'cores': 2, 'active_apps': 0}
llm-chat health: HTTP 200


In [4]:

# A second compact health table for screenshot-friendly evidence.
health_rows = []
for name, url in service_checks.items():
    try:
        response = requests.get(url, timeout=10)
        health_rows.append({'service': name, 'status_code': response.status_code, 'ok': response.ok})
    except Exception as error:
        health_rows.append({'service': name, 'status_code': None, 'ok': False, 'error': str(error)})

for row in health_rows:
    print(row)


{'service': 'aviation-api metrics', 'status_code': 200, 'ok': True}
{'service': 'prometheus readiness', 'status_code': 200, 'ok': True}
{'service': 'grafana health', 'status_code': 200, 'ok': True}
{'service': 'kafka-ui', 'status_code': 200, 'ok': True}
{'service': 'mlflow', 'status_code': 200, 'ok': True}
{'service': 'spark-master json', 'status_code': 200, 'ok': True}
{'service': 'llm-chat health', 'status_code': 200, 'ok': True}


## 3. API Prediction With A Real Gold Row

This proves that the deployed BentoML API can score a historical feature row produced by the Spark Gold table.


In [5]:
run('python -m spark_jobs.call_api_with_gold_sample --year 2024 --month 1 --api-url http://aviation-api:3000/predict', timeout=180)

$ python -m spark_jobs.call_api_with_gold_sample --year 2024 --month 1 --api-url http://aviation-api:3000/predict
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 08:33:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/22 08:33:22 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/22 08:33:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                          (0 + 1) / 1]

                                                                                

[Stage 1:>                       

'WARNING: Using incubator modules: jdk.incubator.vector\nUsing Spark\'s default log4j profile: org/apache/spark/log4j2-defaults.properties\nSetting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n26/06/22 08:33:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\n26/06/22 08:33:22 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).\n26/06/22 08:33:31 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties\n\n[Stage 0:>                                                          (0 + 1) / 1]\n\n                                                                                \n\n[Stage 1:>                                                          (0 + 1) / 

## 4. Dataset Streaming Replay

This is the main dataset-only streaming proof. The script reads historical Gold feature rows from the downloaded lakehouse, publishes each replay event to Kafka topic `simulation.prediction.requests` for streaming evidence, separately calls the deployed API for scoring, and writes JSONL evidence for review. The API does not consume from Kafka in this demo.


In [6]:
simulation_path = STREAM_DIR / 'notebook_gold_simulation.jsonl'
if simulation_path.exists():
    simulation_path.unlink()

cmd = (
    'python -m api.simulate_gold_stream_predict '
    '--year 2024 '
    '--month 1 '
    '--limit 25 '
    '--delay-seconds 0.1 '
    f'--output-jsonl {simulation_path} '
    '--api-url http://aviation-api:3000/predict'
)
run(cmd, timeout=300)

$ python -m api.simulate_gold_stream_predict --year 2024 --month 1 --limit 25 --delay-seconds 0.1 --output-jsonl /workspace/data/local_cache/streaming_predictions/notebook_gold_simulation.jsonl --api-url http://aviation-api:3000/predict
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/22 08:33:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/06/22 08:33:45 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).
26/06/22 08:33:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties

[Stage 0:>                                                          (0 + 1

'WARNING: Using incubator modules: jdk.incubator.vector\nUsing Spark\'s default log4j profile: org/apache/spark/log4j2-defaults.properties\nSetting default log level to "WARN".\nTo adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).\n26/06/22 08:33:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable\n26/06/22 08:33:45 WARN SparkConf: Note that spark.local.dir will be overridden by the value set by the cluster manager (via SPARK_LOCAL_DIRS in standalone/kubernetes and LOCAL_DIRS in YARN).\n26/06/22 08:33:54 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties\n\n[Stage 0:>                                                          (0 + 1) / 1]\n\n                                                                                \n\n[Stage 1:>                                                          (0 + 2) / 

In [7]:
events = tail_jsonl(simulation_path, limit=5)
for event in events:
    source = event.get('source_event', {})
    response = event.get('api_response', {})
    print({
        'sequence': event.get('sequence'),
        'flight_date': source.get('flight_date'),
        'route': f"{source.get('origin')}->{source.get('destination')}",
        'actual_label': source.get('label'),
        'prediction': response.get('prediction'),
        'risk_band': response.get('risk_band'),
        'probability': response.get('disruption_probability'),
    })

{'sequence': 21, 'flight_date': '2024-01-01', 'route': 'ABQ->BUR', 'actual_label': 0.0, 'prediction': 0, 'risk_band': 'low', 'probability': 0.457649}
{'sequence': 22, 'flight_date': '2024-01-01', 'route': 'ABQ->BWI', 'actual_label': 1.0, 'prediction': 1, 'risk_band': 'high', 'probability': 0.503831}
{'sequence': 23, 'flight_date': '2024-01-01', 'route': 'ABQ->DAL', 'actual_label': 0.0, 'prediction': 0, 'risk_band': 'low', 'probability': 0.455362}
{'sequence': 24, 'flight_date': '2024-01-01', 'route': 'ABQ->DAL', 'actual_label': 0.0, 'prediction': 1, 'risk_band': 'high', 'probability': 0.5403}
{'sequence': 25, 'flight_date': '2024-01-01', 'route': 'ABQ->DAL', 'actual_label': 0.0, 'prediction': 0, 'risk_band': 'low', 'probability': 0.372944}


## 5. Kafka Topic Evidence

Open Kafka UI at `http://localhost:8085` and show topic `simulation.prediction.requests`. The cell below checks the same topic through the Kafka Python client, so it works from the Jupyter container without host Docker CLI access.


In [8]:

from kafka import KafkaAdminClient

admin = KafkaAdminClient(
    bootstrap_servers=os.environ.get('KAFKA_BOOTSTRAP_SERVERS', 'kafka:9092'),
    client_id='final-demo-topic-check',
)
topics = sorted(admin.list_topics())
admin.close()
print('Kafka topics:', topics)
print('Expected replay topic present:', 'simulation.prediction.requests' in topics)


Kafka topics: ['simulation.prediction.requests']
Expected replay topic present: True


## 6. 10x API Load Test

This demonstrates the required concurrent API stability check.


In [9]:
run('python3 api/load_test.py --url http://aviation-api:3000/predict --requests 100 --concurrency 10', timeout=180)

$ python3 api/load_test.py --url http://aviation-api:3000/predict --requests 100 --concurrency 10
Successful requests: 100/100
Concurrency: 10x
Mean latency ms: 41.17
Max latency ms: 53.00



'Successful requests: 100/100\nConcurrency: 10x\nMean latency ms: 41.17\nMax latency ms: 53.00\n'

## 7. Prometheus Metrics

These queries confirm that Prometheus observes prediction requests, source labels, latency, failures, and dataset-replay alert states.


In [10]:
def promql(query: str):
    response = requests.get('http://prometheus:9090/api/v1/query', params={'query': query}, timeout=10)
    response.raise_for_status()
    return response.json()['data']['result']

queries = {
    'requests_by_source_and_risk': 'sum by (source, risk_band) (aviation_prediction_requests_by_source_total)',
    'failed_requests': 'sum(bentoml_service_request_total{http_response_code!="200"})',
    'p95_latency_seconds': 'histogram_quantile(0.95, sum(rate(aviation_prediction_latency_seconds_bucket[1m])) by (le))',
    'active_dataset_replay_alerts': 'sum(ALERTS{alertname=~"AviationDatasetReplayInactive|AviationDatasetReplayHighLatency|AviationDatasetReplayAPIFailures|AviationHighRiskBandShare",alertstate="firing"})',
    'dataset_replay_alert_states': 'ALERTS{alertname=~"AviationDatasetReplayInactive|AviationDatasetReplayHighLatency|AviationDatasetReplayAPIFailures|AviationHighRiskBandShare"}',
}

for name, query in queries.items():
    print('---', name, '---')
    print(json.dumps(promql(query), indent=2))


--- requests_by_source_and_risk ---
[
  {
    "metric": {
      "risk_band": "low",
      "source": "dataset_simulation"
    },
    "value": [
      1782117249.016,
      "120"
    ]
  },
  {
    "metric": {
      "risk_band": "high",
      "source": "direct_api"
    },
    "value": [
      1782117249.016,
      "507"
    ]
  },
  {
    "metric": {
      "risk_band": "high",
      "source": "dataset_simulation"
    },
    "value": [
      1782117249.016,
      "30"
    ]
  }
]
--- failed_requests ---
[
  {
    "metric": {},
    "value": [
      1782117249.021,
      "0"
    ]
  }
]
--- p95_latency_seconds ---
[
  {
    "metric": {},
    "value": [
      1782117249.025,
      "0.00475"
    ]
  }
]
--- active_dataset_replay_alerts ---
[]
--- dataset_replay_alert_states ---
[]


## 8. Grafana And Service Dashboards

Open Grafana at `http://localhost:3001` and show the merged final dashboard:

### Aviation Final Demo Dashboard

Show total prediction requests, dataset replay records scored, overall and replay request rates, failed API requests, latest dataset probability, p50/p95 latency, risk-band splits, probability trend, probability distribution, prediction requests by source, active dataset replay alerts, and replay alert states. The alerts are scoped to the simulated dataset replay: inactivity, high replay scoring latency, replay-window API failures, and high-risk replay share.

Do not rebuild MLflow, Airflow, Spark, or Kafka evidence inside Grafana. Use their own service UIs:

- MLflow: registered model, AUC, positive precision, positive recall, confusion metrics, model artifact, staging/production alias.
- Airflow: DAG graph and run status.
- Spark UI: Spark applications, jobs, and stages.
- Kafka UI: `simulation.prediction.requests` topic messages.


## 9. LLM Project Q&A Assistant

The optional LLM layer uses Gemini 3.5 Flash when `GEMINI_API_KEY` is configured. It acts as a project/codebase/status assistant. It can reference bounded source/docs/notebook excerpts, dashboard JSON, dataset replay evidence, Prometheus metrics, reachable service APIs for MLflow, Airflow, Kafka, MinIO, and Spark, plus bounded recent service log tails. If the key is missing, it returns a local fallback summary.


In [11]:
question = 'Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.'
cmd = (
    'python -m api.llm_ops_assistant '
    '--prometheus-url http://prometheus:9090 '
    f'--simulation-jsonl {simulation_path} '
    f'--question {json.dumps(question)}'
)
run(cmd, timeout=180)

$ python -m api.llm_ops_assistant --prometheus-url http://prometheus:9090 --simulation-jsonl /workspace/data/local_cache/streaming_predictions/notebook_gold_simulation.jsonl --question "Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations."
{
  "provider": "local_fallback",
  "model": "gemini-3.5-flash",
  "question": "Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.",
  "answer": "GEMINI_API_KEY is not configured, so this is the local fallback summary.\\n\\nQuestion: Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.\\n\\nDataset streaming replay status: route=ABQ->DAL, risk=low, probability=0.372944.\\nThe deployed model is working through BentoML, and

'{\n  "provider": "local_fallback",\n  "model": "gemini-3.5-flash",\n  "question": "Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.",\n  "answer": "GEMINI_API_KEY is not configured, so this is the local fallback summary.\\\\n\\\\nQuestion: Explain what is happening in this aviation dataset streaming demo right now. Mention the downloaded datasets, replay simulation, risk predictions, and limitations.\\\\n\\\\nDataset streaming replay status: route=ABQ->DAL, risk=low, probability=0.372944.\\\\nThe deployed model is working through BentoML, and Prometheus/Grafana track request sources, latency, failures, and risk bands.\\n\\n[Gemini unavailable: HTTPSConnectionPool(host=\'generativelanguage.googleapis.com\', port=443): Read timed out. (read timeout=45)]"\n}\n'

## 10. Browser Chat UI

Start the browser assistant with `docker compose up -d llm-chat`, then open `http://localhost:7860`. Ask:

- `Which files implement the dataset replay and API scoring path?`
- `How does the dataset streaming replay work?`
- `What is the current status of MLflow, Kafka, MinIO, Spark, and Grafana?`
- `Which downloaded datasets are used by the model?`
- `What should I show in Grafana for final presentation?`


In [12]:
try:
    response = requests.get('http://llm-chat:7860/api/health', timeout=10)
    print(response.json())
    print('Browser URL: http://localhost:7860')
except Exception as error:
    print('LLM chat UI is not running yet. Start it with: docker compose up -d llm-chat')
    print(error)

{'ok': True, 'provider': 'gemini', 'model': 'gemini-3.5-flash', 'gemini_key_configured': True, 'scope': ['repo source/docs/notebooks', 'MLflow REST', 'Airflow REST', 'Kafka topics/messages', 'MinIO bucket/object samples', 'Spark UI REST', 'Grafana dashboard JSON', 'Docker service log tails', 'Prometheus metrics', 'streaming replay JSONL']}
Browser URL: http://localhost:7860


## Final Talking Points

- The model is trained from downloaded ARCO-ERA5 weather and BTS flight outcomes.
- The final demo runs a historical test-set replay from downloaded Gold features.
- Kafka shows event movement, BentoML scores each event, and Grafana/Prometheus show operational evidence.
- The LLM assistant is a Q&A layer over project evidence; it is not used for model training or scoring.
